# Notebook 67 — Residual Symbolic Correction Layer

Notebook 66 showed that the sparse symbolic chart beats or matches flexible ML baselines under hard regime shift, but the remaining error suggests the governing field may be slightly incomplete.

Base field:

\[
g_0(r,c;\beta)=
\beta_0+\beta_c c+\beta_r r+\beta_{c^3}c^3+\beta_{r^2}r^2+\beta_{rc^2}rc^2
\]

Notebook 67 asks:

> Can the remaining residual error be explained by a small, stable symbolic correction layer?

\[
g(r,c;\beta,\gamma)=g_0(r,c;\beta)+h(r,c;\gamma)
\]

Candidate correction terms include:

\[
rc,\; c^2,\; r^3,\; r^2c,\; c^2r,\; c^4,\; r^4,\; \sin(c),\; \cos(c),\; r\sin(c)
\]

Main goal: identify whether residual error has a stable symbolic form, without adding flexible ML complexity.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os, glob, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LassoCV, Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, LeaveOneOut

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    Markdown = lambda x: x

np.random.seed(42)

OUTPUT_DIR = "paper67_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})

## 1. Load data or synthetic fallback

In [ ]:
DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet", "*residual*flow*.csv",
        "*governing*flow*.parquet", "*governing*flow*.csv",
        "*.parquet", "*.csv", "*.pkl", "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.05, 5: 0.02, 7: 0.06}[k]
                        flow_shift = 0.05 if flow_mode == "nonlinear" else -0.02
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift + flow_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if task == "zeta_vs_poisson":
                                    g -= 0.015 * c
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append({
                                    "system": system, "task": task, "forcing_mode": forcing_mode,
                                    "k": k, "flow_mode": flow_mode,
                                    "condition_coord": c, "residual": r,
                                    "predicted_flow": g + noise, "next_residual": next_r,
                                    "delta_condition": delta_c, "sample_id": sample_id,
                                    "path_id": path_id, "window_id": window_id,
                                })
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

def align_schema(df):
    alias_groups = {
        "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
        "residual": ["residual", "r", "resid"],
        "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
        "next_residual": ["next_residual", "r_next", "next_r"],
        "delta_condition": ["delta_condition", "dc", "delta_c"],
        "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
    }
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        df["delta_condition"] = np.gradient(df["condition_coord"].to_numpy())

    defaults = {
        "system": "unknown_system", "task": "unknown_task",
        "forcing_mode": "unknown_forcing", "k": 5,
        "flow_mode": "unknown_flow", "sample_id": np.arange(len(df)),
        "path_id": 0, "window_id": np.arange(len(df)),
    }
    for key, value in defaults.items():
        if key not in df.columns:
            df[key] = value

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")
    return df

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Synthetic fallback:", USE_SYNTHETIC_FALLBACK)
print("Rows:", len(df))
display(df.head())

## 2. Base field, correction terms, and coefficient table

In [ ]:
BASE_TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in BASE_TERM_NAMES]

def base_design(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([np.ones_like(r), c, r, c**3, r**2, r * c**2])

def correction_design(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return pd.DataFrame({
        "rc": r * c,
        "c^2": c**2,
        "r^3": r**3,
        "r^2 c": (r**2) * c,
        "c^2 r": (c**2) * r,
        "c^4": c**4,
        "r^4": r**4,
        "sin_c": np.sin(c),
        "cos_c": np.cos(c),
        "tanh_c": np.tanh(c),
        "r sin_c": r * np.sin(c),
    })

def governing_field_base(sub, beta):
    return base_design(sub) @ np.asarray(beta, dtype=float)

def fit_base_template(sub, alpha=1e-6):
    X = base_design(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta
    stats = {"n": len(sub), "template_rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(BASE_TERM_NAMES, beta):
        stats[f"coef_{name}"] = float(coef)
    return beta, pred, stats

def trajectory_gap_beta(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i + 1] - cgrid[i])
            row = pd.DataFrame({"residual": [r], "condition_coord": [c]})
            g = float(np.clip(governing_field_base(row, beta)[0], -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    return float(np.mean([
        math.sqrt(mean_squared_error(integrate(beta_ref, r0), integrate(beta_cmp, r0)))
        for r0 in r0s
    ]))

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
regime_rows = []
regime_subsets = {}

for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, stats = fit_base_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id, "system": vals[0], "task": vals[1],
        "forcing_mode": vals[2], "k": float(vals[3]), "flow_mode": vals[4],
    }
    row.update(stats)
    regime_rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(regime_rows).reset_index(drop=True)
coef_df.to_csv(os.path.join(OUTPUT_DIR, "base_coefficient_table.csv"), index=False)
print("Regime count:", len(coef_df))
display(coef_df.head())

## 3. Base pruned symbolic coefficient chart

In [ ]:
def build_symbolic_features(df_in, columns=None, reduced_terms=None):
    X = pd.get_dummies(df_in[["forcing_mode", "flow_mode", "system", "task"]], drop_first=False)
    X["k"] = df_in["k"].astype(float).values
    X["k2"] = df_in["k"].astype(float).values ** 2

    ff = df_in["forcing_mode"].astype(str) + "__x__" + df_in["flow_mode"].astype(str)
    X_ff = pd.get_dummies(ff, prefix="ff")
    sf = df_in["system"].astype(str) + "__x__" + df_in["forcing_mode"].astype(str)
    X_sf = pd.get_dummies(sf, prefix="sf")
    X_fk = pd.get_dummies(df_in["forcing_mode"].astype(str), prefix="fk").multiply(
        df_in["k"].astype(float).to_numpy().reshape(-1, 1)
    )
    X = pd.concat([X, X_ff, X_sf, X_fk], axis=1)

    if reduced_terms is not None:
        X = X.reindex(columns=reduced_terms, fill_value=0.0)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X.astype(float)

def term_stability_table(df_in, n_splits=8, threshold=1e-4):
    n = len(df_in)
    splitter = LeaveOneOut() if n <= 12 else KFold(n_splits=min(n_splits, n), shuffle=True, random_state=42)
    all_features = build_symbolic_features(df_in).columns.tolist()
    stability = {coef: {feat: 0 for feat in all_features} for coef in COEF_COLS}
    fold_count = 0

    for train_idx, _ in splitter.split(df_in):
        train_df = df_in.iloc[train_idx].reset_index(drop=True)
        X_train = build_symbolic_features(train_df, columns=all_features)
        for coef in COEF_COLS:
            y = train_df[coef].to_numpy(dtype=float)
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X_train)
            model = LassoCV(cv=min(5, len(train_df)), random_state=fold_count + 1, max_iter=30000)
            model.fit(Xs, y)
            for feat, val in zip(all_features, model.coef_):
                if abs(val) > threshold:
                    stability[coef][feat] += 1
        fold_count += 1

    rows = []
    for coef in COEF_COLS:
        for feat in all_features:
            rows.append({
                "coefficient": coef, "term": feat,
                "frequency": stability[coef][feat] / fold_count,
                "count": stability[coef][feat], "folds": fold_count,
            })
    return pd.DataFrame(rows)

stability_df = term_stability_table(coef_df)
STABILITY_THRESHOLD = 0.5
stable_terms_by_coef = {}
for coef in COEF_COLS:
    sub = stability_df[(stability_df["coefficient"] == coef) & (stability_df["frequency"] >= STABILITY_THRESHOLD)]
    stable_terms_by_coef[coef] = sub["term"].tolist()

def predict_pruned_beta(train_df, test_df, terms_by_coef):
    Yhat = np.zeros((len(test_df), len(COEF_COLS)), dtype=float)
    for j, coef in enumerate(COEF_COLS):
        terms = terms_by_coef.get(coef, [])
        if len(terms) == 0:
            Yhat[:, j] = train_df[coef].mean()
            continue
        X_train = build_symbolic_features(train_df, reduced_terms=terms)
        X_test = build_symbolic_features(test_df, columns=X_train.columns, reduced_terms=terms)
        y_train = train_df[coef].to_numpy(dtype=float)
        scaler = StandardScaler()
        Xtr = scaler.fit_transform(X_train)
        Xte = scaler.transform(X_test)
        model = Ridge(alpha=1.0)
        model.fit(Xtr, y_train)
        Yhat[:, j] = model.predict(Xte)
    return Yhat

display(pd.DataFrame([
    {"coefficient": coef, "n_terms": len(terms), "terms": ", ".join(terms)}
    for coef, terms in stable_terms_by_coef.items()
]))

## 4. Build pointwise residual dataset from base symbolic chart

In [ ]:
hard_block_masks = {
    "k_high": coef_df["k"].astype(float) == 7.0,
    "k_low": coef_df["k"].astype(float) == 3.0,
    "forcing::condition": coef_df["forcing_mode"].astype(str) == "condition_gap",
    "forcing::capacity": coef_df["forcing_mode"].astype(str) == "capacity_gap",
    "forcing::feature": coef_df["forcing_mode"].astype(str) == "feature_gap",
    "system::entropy": coef_df["system"].astype(str) == "entropy",
    "system::unevenness": coef_df["system"].astype(str) == "unevenness",
    "flow::linear": coef_df["flow_mode"].astype(str) == "linear",
    "flow::nonlinear": coef_df["flow_mode"].astype(str) == "nonlinear",
}

residual_rows = []
base_eval_rows = []

for block_name, test_mask in hard_block_masks.items():
    train_coef = coef_df.loc[~test_mask].reset_index(drop=True)
    test_coef = coef_df.loc[test_mask].reset_index(drop=True)
    if len(test_coef) == 0 or len(train_coef) < 5:
        continue
    beta_hat_mat = predict_pruned_beta(train_coef, test_coef, stable_terms_by_coef)

    for i, rid in enumerate(test_coef["regime_id"].tolist()):
        sub = regime_subsets[rid].copy()
        beta_true = test_coef.loc[i, COEF_COLS].to_numpy(dtype=float)
        beta_hat = beta_hat_mat[i]
        y_true = sub["predicted_flow"].to_numpy(dtype=float)
        y_base = governing_field_base(sub, beta_hat)
        err = y_true - y_base

        base_eval_rows.append({
            "block": block_name, "regime_id": rid, "model": "base_pruned_symbolic",
            "coef_rmse": math.sqrt(mean_squared_error(beta_true, beta_hat)),
            "field_rmse": math.sqrt(mean_squared_error(y_true, y_base)),
            "traj_rmse": trajectory_gap_beta(sub, beta_true, beta_hat),
        })

        tmp = sub[["condition_coord", "residual"]].copy()
        tmp["target_error"] = err
        tmp["true_flow"] = y_true
        tmp["base_pred_flow"] = y_base
        tmp["block"] = block_name
        tmp["regime_id"] = rid
        tmp["system"] = test_coef.loc[i, "system"]
        tmp["task"] = test_coef.loc[i, "task"]
        tmp["forcing_mode"] = test_coef.loc[i, "forcing_mode"]
        tmp["k"] = test_coef.loc[i, "k"]
        tmp["flow_mode"] = test_coef.loc[i, "flow_mode"]
        residual_rows.append(tmp)

residual_df = pd.concat(residual_rows, ignore_index=True)
base_eval_df = pd.DataFrame(base_eval_rows)

print("Residual correction rows:", residual_df.shape)
display(residual_df.head())
display(base_eval_df.groupby("block")[["field_rmse", "traj_rmse"]].mean())
residual_df.to_csv(os.path.join(OUTPUT_DIR, "pointwise_base_residuals.csv"), index=False)

## 5. Candidate residual correction terms and stability

In [ ]:
CORRECTION_TERMS = correction_design(residual_df).columns.tolist()
print("Correction terms:", CORRECTION_TERMS)

def residual_correction_matrix(data, columns=None):
    X = correction_design(data)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0.0)
    return X.astype(float)

def fit_correction_model(train_resid_df, columns=None, alpha=None):
    X = residual_correction_matrix(train_resid_df, columns=columns)
    y = train_resid_df["target_error"].to_numpy(dtype=float)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    cv = min(5, max(2, len(train_resid_df)//200))
    if alpha is None:
        model = LassoCV(cv=cv, random_state=42, max_iter=30000)
    else:
        model = Lasso(alpha=alpha, random_state=42, max_iter=30000)
    model.fit(Xs, y)
    active = [(feat, val) for feat, val in zip(X.columns, model.coef_) if abs(val) > 1e-5]
    return {"model": model, "scaler": scaler, "columns": X.columns.tolist(), "active": active}

def predict_correction(pack, data):
    X = residual_correction_matrix(data, columns=pack["columns"])
    Xs = pack["scaler"].transform(X)
    return pack["model"].predict(Xs)

def correction_stability_by_block(resid_df):
    blocks = sorted(resid_df["block"].unique())
    counts = {term: 0 for term in CORRECTION_TERMS}
    fold_count = 0
    for holdout_block in blocks:
        train = resid_df[resid_df["block"] != holdout_block].reset_index(drop=True)
        if len(train) < 50:
            continue
        pack = fit_correction_model(train, columns=CORRECTION_TERMS)
        active_terms = set([t for t, v in pack["active"]])
        for term in CORRECTION_TERMS:
            if term in active_terms:
                counts[term] += 1
        fold_count += 1
    return pd.DataFrame([
        {"term": term, "count": counts[term], "folds": fold_count, "frequency": counts[term] / max(fold_count, 1)}
        for term in CORRECTION_TERMS
    ]).sort_values("frequency", ascending=False)

corr_stability_df = correction_stability_by_block(residual_df)
display(corr_stability_df)
corr_stability_df.to_csv(os.path.join(OUTPUT_DIR, "correction_term_stability.csv"), index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(corr_stability_df["term"], corr_stability_df["frequency"])
ax.set_ylim(0, 1.05)
ax.set_ylabel("selection frequency")
ax.set_title("Residual correction term stability")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure1_correction_term_stability.png"), dpi=220, bbox_inches="tight")
plt.show()

CORRECTION_STABILITY_THRESHOLD = 0.5
stable_correction_terms = corr_stability_df[corr_stability_df["frequency"] >= CORRECTION_STABILITY_THRESHOLD]["term"].tolist()
if len(stable_correction_terms) == 0:
    stable_correction_terms = corr_stability_df.head(3)["term"].tolist()
print("Stable correction terms:", stable_correction_terms)

## 6. Hard-block evaluation: base vs corrected field

In [ ]:
corrected_eval_rows = []

for block_name, test_mask in hard_block_masks.items():
    train_coef = coef_df.loc[~test_mask].reset_index(drop=True)
    test_coef = coef_df.loc[test_mask].reset_index(drop=True)
    if len(test_coef) == 0 or len(train_coef) < 5:
        continue

    beta_hat_mat = predict_pruned_beta(train_coef, test_coef, stable_terms_by_coef)
    train_resid = residual_df[residual_df["block"] != block_name].reset_index(drop=True)
    correction_pack_all = fit_correction_model(train_resid, columns=CORRECTION_TERMS)
    correction_pack_stable = fit_correction_model(train_resid, columns=stable_correction_terms)

    for i, rid in enumerate(test_coef["regime_id"].tolist()):
        sub = regime_subsets[rid].copy()
        beta_hat = beta_hat_mat[i]
        y_true = sub["predicted_flow"].to_numpy(dtype=float)
        y_base = governing_field_base(sub, beta_hat)
        y_corr_all = y_base + predict_correction(correction_pack_all, sub)
        y_corr_stable = y_base + predict_correction(correction_pack_stable, sub)

        for model_name, pred in {
            "base_pruned_symbolic": y_base,
            "corrected_all_terms": y_corr_all,
            "corrected_stable_terms": y_corr_stable,
        }.items():
            corrected_eval_rows.append({
                "block": block_name,
                "regime_id": rid,
                "model": model_name,
                "field_rmse": math.sqrt(mean_squared_error(y_true, pred)),
                "traj_proxy_rmse": math.sqrt(mean_squared_error(y_true, pred)),
            })

corrected_eval_df = pd.DataFrame(corrected_eval_rows)
base_lookup = corrected_eval_df[corrected_eval_df["model"] == "base_pruned_symbolic"].set_index(["block", "regime_id"])["field_rmse"]
corrected_eval_df["field_improvement_vs_base"] = [
    base_lookup.loc[(row["block"], row["regime_id"])] - row["field_rmse"]
    for _, row in corrected_eval_df.iterrows()
]

corrected_summary_df = corrected_eval_df.groupby(["block", "model"])[
    ["field_rmse", "field_improvement_vs_base", "traj_proxy_rmse"]
].mean().reset_index()

display(corrected_summary_df.sort_values(["block", "field_rmse"]))
corrected_summary_df.to_csv(os.path.join(OUTPUT_DIR, "corrected_field_summary_by_block.csv"), index=False)
corrected_eval_df.to_csv(os.path.join(OUTPUT_DIR, "corrected_field_eval_raw.csv"), index=False)

## 7. Correction usefulness figures

In [ ]:
pivot = corrected_summary_df.pivot(index="block", columns="model", values="field_rmse")
fig, ax = plt.subplots(figsize=(12, 4.8))
pivot.plot(kind="bar", ax=ax)
ax.set_ylabel("field RMSE")
ax.set_title("Base vs corrected symbolic field")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure2_base_vs_corrected_field_rmse.png"), dpi=220, bbox_inches="tight")
plt.show()

pivot_imp = corrected_summary_df[corrected_summary_df["model"] != "base_pruned_symbolic"].pivot(
    index="block", columns="model", values="field_improvement_vs_base"
)
fig, ax = plt.subplots(figsize=(12, 4.8))
pivot_imp.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("base RMSE - corrected RMSE")
ax.set_title("Correction improvement over base")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure3_correction_improvement.png"), dpi=220, bbox_inches="tight")
plt.show()

global_corr_summary = corrected_eval_df.groupby("model")[["field_rmse", "field_improvement_vs_base"]].mean().reset_index()
display(global_corr_summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(global_corr_summary["model"], global_corr_summary["field_rmse"])
ax.set_ylabel("mean field RMSE")
ax.set_title("Global correction comparison")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure4_global_correction_comparison.png"), dpi=220, bbox_inches="tight")
plt.show()

## 8. Representative residual correction diagnostic

In [ ]:
stable_rows = corrected_eval_df[corrected_eval_df["model"] == "corrected_stable_terms"].copy()
best_help = stable_rows.sort_values("field_improvement_vs_base", ascending=False).iloc[0]
rep_block = best_help["block"]
rep_rid = best_help["regime_id"]

test_mask = hard_block_masks[rep_block]
train_coef = coef_df.loc[~test_mask].reset_index(drop=True)
test_coef = coef_df.loc[test_mask].reset_index(drop=True)
test_row = test_coef[test_coef["regime_id"] == rep_rid].reset_index(drop=True)

beta_hat = predict_pruned_beta(train_coef, test_row, stable_terms_by_coef)[0]
sub = regime_subsets[rep_rid].copy()
y_true = sub["predicted_flow"].to_numpy(dtype=float)
y_base = governing_field_base(sub, beta_hat)

train_resid = residual_df[residual_df["block"] != rep_block].reset_index(drop=True)
pack_stable = fit_correction_model(train_resid, columns=stable_correction_terms)
y_corr = y_base + predict_correction(pack_stable, sub)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
panels = [
    ("True flow", y_true),
    ("Base symbolic", y_base),
    ("Base residual", y_true - y_base),
    ("Corrected residual", y_true - y_corr),
]
for ax, (title, vals) in zip(axes, panels):
    sc = ax.scatter(sub["condition_coord"], sub["residual"], c=vals, s=16)
    ax.set_title(title)
    ax.set_xlabel("condition c")
    plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
axes[0].set_ylabel("residual r")
fig.suptitle(f"Residual correction diagnostic: {rep_rid}", y=1.03)
plt.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figure5_residual_correction_diagnostic.png"), dpi=220, bbox_inches="tight")
plt.show()

## 9. Final correction equation and decision rule

In [ ]:
final_corr_pack = fit_correction_model(residual_df, columns=stable_correction_terms)

def format_correction_equation(active_terms, intercept):
    parts = [f"h(r,c) = {intercept:.6f}"]
    for feat, val in active_terms:
        sign = "+" if val >= 0 else "-"
        parts.append(f"{sign} {abs(val):.6f}·{feat}")
    return " ".join(parts)

corr_eq = format_correction_equation(final_corr_pack["active"], final_corr_pack["model"].intercept_)
print(corr_eq)

corr_eq_df = pd.DataFrame([{
    "correction_terms": ", ".join(stable_correction_terms),
    "equation": corr_eq,
    "n_active": len(final_corr_pack["active"]),
}])
display(corr_eq_df)
corr_eq_df.to_csv(os.path.join(OUTPUT_DIR, "final_correction_equation.csv"), index=False)

decision_rows = []
base_global = global_corr_summary[global_corr_summary["model"] == "base_pruned_symbolic"]["field_rmse"].iloc[0]
for model in ["corrected_all_terms", "corrected_stable_terms"]:
    sub = corrected_summary_df[corrected_summary_df["model"] == model]
    global_rmse = global_corr_summary[global_corr_summary["model"] == model]["field_rmse"].iloc[0]
    helped_blocks = int((sub["field_improvement_vs_base"] > 0).sum())
    hurt_blocks = int((sub["field_improvement_vs_base"] < 0).sum())
    total_blocks = int(sub["block"].nunique())
    if global_rmse < base_global and helped_blocks >= hurt_blocks:
        verdict = "keep correction layer"
    elif global_rmse < base_global:
        verdict = "optional correction; improves global error but mixed by block"
    else:
        verdict = "do not keep correction as final model"
    decision_rows.append({
        "model": model,
        "base_global_field_rmse": base_global,
        "corrected_global_field_rmse": global_rmse,
        "global_improvement": base_global - global_rmse,
        "mean_block_improvement": sub["field_improvement_vs_base"].mean(),
        "helped_blocks": helped_blocks,
        "hurt_blocks": hurt_blocks,
        "total_blocks": total_blocks,
        "verdict": verdict,
    })

decision_df = pd.DataFrame(decision_rows)
display(decision_df)
decision_df.to_csv(os.path.join(OUTPUT_DIR, "correction_decision_table.csv"), index=False)

## 10. Paper-ready paragraph and export manifest

In [ ]:
best_corr = decision_df.sort_values("corrected_global_field_rmse").iloc[0]
stable_decision = decision_df[decision_df["model"] == "corrected_stable_terms"].iloc[0]

paragraph = f'''
## Residual symbolic correction

After fixing the shared governing field and sparse symbolic coefficient chart, we tested whether the remaining pointwise flow error contained stable symbolic structure. Candidate correction terms were fit to the residual `true_flow - base_symbolic_flow` using block-wise stability selection. The stable correction set was `{", ".join(stable_correction_terms)}`. The best corrected model changed global field RMSE from {best_corr["base_global_field_rmse"]:.4f} to {best_corr["corrected_global_field_rmse"]:.4f}, with mean block improvement {best_corr["mean_block_improvement"]:.4f}. For the stable correction layer, the verdict was: `{stable_decision["verdict"]}`. This test separates equation completeness from model class choice: if the correction layer improves consistently, the governing field should be expanded; if not, the base field remains the final interpretable law and the residual is treated as regime/noise-limited.
'''
with open(os.path.join(OUTPUT_DIR, "residual_correction_paragraph.md"), "w", encoding="utf-8") as f:
    f.write(paragraph.strip() + "\\n")
display(Markdown(paragraph))

manifest = {
    "data_source": data_source,
    "synthetic_fallback": bool(USE_SYNTHETIC_FALLBACK),
    "base_terms": BASE_TERM_NAMES,
    "candidate_correction_terms": CORRECTION_TERMS,
    "stable_correction_terms": stable_correction_terms,
    "outputs": sorted(os.listdir(OUTPUT_DIR)),
}
with open(os.path.join(OUTPUT_DIR, "manifest.json"), "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

display(pd.DataFrame({"output_file": sorted(os.listdir(OUTPUT_DIR))}))
print("Saved outputs under:", OUTPUT_DIR)

## Final statement

Notebook 67 tests equation completeness rather than model-class competition.

If correction terms consistently improve hard-block field RMSE, update the governing field to:

\[
g(r,c;\beta,\gamma)=g_0(r,c;\beta)+h(r,c;\gamma).
\]

If correction terms are unstable or mixed, retain the base field and report that remaining error is regime/noise-limited rather than a stable missing symbolic term.